# Python: Structuring Larger Projects with Modern Packaging

**Modules covered:**
1. From Scripts to Scalable Systems
2. Dependency Management and Security Scanning
3. Packaging and Advanced Distribution
4. CI/CD for Scalable Project Workflows
5. Quality Gates and Type Safety at Scale
6. Extensibility and Plug-in Architectures

> **Note:** This is a demo-based course. The cells below combine the key concepts from the slides with runnable examples.

---
# Module 1 — From Scripts to Scalable Systems

## 1. Why Structure Matters

Most Python projects start as one small script. Over time, without deliberate structure, they accumulate these problems:

| Problem | Description |
|---|---|
| No packages | Code lives as loose `.py` files at the root |
| No namespace | Name collisions become likely as the project grows |
| No separation of concerns | Logic, I/O, and config are mixed in the same file |
| Growing pile of files | Navigation and maintenance become painful |

Projects collapse not because the team did something wrong, but because **the structure never evolved**.

In [1]:
# Anti-pattern: everything in one flat file

def load_customer(customer_id):
    return {"id": customer_id, "name": "Acme Corp", "email": "acme@example.com"}

def generate_invoice(customer, amount):
    return {"customer": customer["name"], "amount": amount, "status": "pending"}

def send_email(customer, invoice):
    print(f"Sending invoice of ${invoice['amount']} to {customer['email']}")

customer = load_customer(42)
invoice  = generate_invoice(customer, 1500)
send_email(customer, invoice)

Sending invoice of $1500 to acme@example.com


---
## 2. Common Layout Patterns

### The `src` Layout

```
my_project/
├── src/
│   └── my_package/
│       ├── __init__.py
│       ├── customers.py
│       ├── invoices.py
│       └── emails.py
├── tests/
│   └── test_invoices.py
├── pyproject.toml
└── README.md
```

| Benefit | Explanation |
|---|---|
| Prevents accidental imports from root | Python cannot import `my_package` unless installed |
| Forces package installation before usage | `pip install -e .` — mirrors production behaviour |
| Tools behave consistently | Linters, type checkers, test runners all agree |
| Sets up packaging and distribution | Layout maps directly to `pyproject.toml` |

> Large organisations rely on this pattern to distribute responsibilities across teams.

In [2]:
layout = {
    "src/my_package/__init__.py": "",
    "src/my_package/customers.py": "def load(customer_id): ...",
    "src/my_package/invoices.py": "def generate(customer, amount): ...",
    "src/my_package/emails.py": "def send(customer, invoice): ...",
    "tests/test_invoices.py": "from my_package.invoices import generate",
    "pyproject.toml": "[tool.setuptools]\npackage-dir = {'': 'src'}",
}
for path, preview in layout.items():
    print(f"{path:<45} → {preview or '(empty init)'}")

src/my_package/__init__.py                    → (empty init)
src/my_package/customers.py                   → def load(customer_id): ...
src/my_package/invoices.py                    → def generate(customer, amount): ...
src/my_package/emails.py                      → def send(customer, invoice): ...
tests/test_invoices.py                        → from my_package.invoices import generate
pyproject.toml                                → [tool.setuptools]
package-dir = {'': 'src'}


---
## 3. Monorepo vs. Multi-package Strategies

| Dimension | Multi-package | Monorepo |
|---|---|---|
| Team autonomy | High — teams ship independently | Lower — changes affect everyone |
| Versioning | Explicit per-package | Shared or lockstep |
| Cross-project testing | Requires extra tooling | Trivial |
| Refactoring | Harder across repo boundaries | Safer — single atomic commit |
| Coordination overhead | High | Low |

**Multi-package tradeoffs:** versioning complexity, cross-package testing overhead, changes take longer.

**Monorepo advantages:** stronger automation, clear ownership via CODEOWNERS, tooling built for multi-package (e.g. `uv` workspaces).

```
Multi-package if:                 Monorepo if:
─────────────────────────         ──────────────────────────
Teams ship independently          Teams collaborate often
Explicit versioning needed        Teams share tooling
Strong isolation required         Unified code visibility valued
```

> The fictional company *Globomantics* can choose either way — the `src` layout works regardless.

In [3]:
print("""
Monorepo layout:
globomantics/
├── packages/
│   ├── billing/      (src/billing/ + pyproject.toml)
│   ├── notifications/(src/notifications/ + pyproject.toml)
│   └── shared/       (src/shared/ + pyproject.toml)
├── .github/workflows/ci.yml
└── pyproject.toml    ← root workspace config

Multi-package layout:
github.com/globomantics/billing          ← separate repo
github.com/globomantics/notifications    ← separate repo
github.com/globomantics/shared           ← published to PyPI
""")


Monorepo layout:
globomantics/
├── packages/
│   ├── billing/      (src/billing/ + pyproject.toml)
│   ├── notifications/(src/notifications/ + pyproject.toml)
│   └── shared/       (src/shared/ + pyproject.toml)
├── .github/workflows/ci.yml
└── pyproject.toml    ← root workspace config

Multi-package layout:
github.com/globomantics/billing          ← separate repo
github.com/globomantics/notifications    ← separate repo
github.com/globomantics/shared           ← published to PyPI



## Module 1 — Summary

| Concept | Key Takeaway |
|---|---|
| Why structure matters | Unstructured projects collapse under their own growth |
| `src` layout | Prevents import accidents, enforces installation, works with all tools |
| Monorepo | Best when teams collaborate, share tooling, want atomic changes |
| Multi-package | Best when teams are independent, need explicit version isolation |

---
# Module 2 — Dependency Management and Security Scanning

## 4. Virtual Environments and pyproject.toml

A virtual environment isolates a project's Python dependencies from the system installation.

```bash
python -m venv .venv
source .venv/bin/activate      # macOS/Linux
.venv\Scripts\activate         # Windows
pip install -e .               # install in editable mode
pip install -e '.[dev]'        # include dev extras
```

### pyproject.toml — PEP 621 standard (replaces setup.py, setup.cfg, MANIFEST.in)

```toml
[build-system]
requires      = ["setuptools>=68", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name        = "billing"
version     = "1.0.0"
requires-python = ">=3.11"
dependencies = ["requests>=2.31", "pydantic>=2.0"]

[project.optional-dependencies]
dev = ["pytest", "ruff", "mypy"]

[tool.setuptools.packages.find]
where = ["src"]
```

| Section | Purpose |
|---|---|
| `[build-system]` | Declares the build backend |
| `[project]` | Package metadata — name, version, Python requirement |
| `dependencies` | Runtime requirements |
| `[project.optional-dependencies]` | Extras like `dev`, `test`, `docs` |
| `[tool.*]` | Tool-specific config (ruff, mypy, pytest…) in one file |

In [4]:
import tomllib

sample_toml = b"""
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "billing"
version = "1.0.0"
requires-python = ">=3.11"
dependencies = ["requests>=2.31", "pydantic>=2.0"]

[project.optional-dependencies]
dev = ["pytest", "ruff", "mypy"]
"""

config = tomllib.loads(sample_toml.decode())
print("Project      :", config["project"]["name"], config["project"]["version"])
print("Python req.  :", config["project"]["requires-python"])
print("Runtime deps :", config["project"]["dependencies"])
print("Dev deps     :", config["project"]["optional-dependencies"]["dev"])

Project      : billing 1.0.0
Python req.  : >=3.11
Runtime deps : ['requests>=2.31', 'pydantic>=2.0']
Dev deps     : ['pytest', 'ruff', 'mypy']


---
## 5. Managing Dependencies with Poetry and pip-tools

### Poetry — all-in-one tool

```bash
pipx install poetry
poetry new billing
poetry add requests pydantic
poetry add --group dev pytest ruff mypy
poetry install          # creates .venv and installs everything
poetry update           # update deps and regenerate poetry.lock
```

### pip-tools — lightweight lock file compiler

```bash
pip install pip-tools
pip-compile pyproject.toml --output-file requirements.lock
pip-compile pyproject.toml --extra dev --output-file requirements-dev.lock
pip-sync requirements.lock requirements-dev.lock
```

| Feature | Poetry | pip-tools |
|---|---|---|
| Lock file | `poetry.lock` | `requirements.lock` |
| Manages `.venv` | Yes | No |
| Build & publish | Built-in | Not included |
| Learning curve | Higher | Low |
| pyproject.toml native | Yes | Yes (≥ 7) |

In [5]:
abstract_requirements = ["requests>=2.31", "pydantic>=2.0"]

pinned_lock_file = [
    "requests==2.31.0          # via billing",
    "certifi==2024.2.2         # via requests",
    "charset-normalizer==3.3.2 # via requests",
    "idna==3.6                 # via requests",
    "urllib3==2.2.1            # via requests",
    "pydantic==2.6.3           # via billing",
    "pydantic-core==2.16.3     # via pydantic",
]

print("Abstract (pyproject.toml):")
for r in abstract_requirements:
    print(f"  {r}")
print("\nPinned (lock file — every transitive dep fixed):")
for line in pinned_lock_file:
    print(f"  {line}")

Abstract (pyproject.toml):
  requests>=2.31
  pydantic>=2.0

Pinned (lock file — every transitive dep fixed):
  requests==2.31.0          # via billing
  certifi==2024.2.2         # via requests
  charset-normalizer==3.3.2 # via requests
  idna==3.6                 # via requests
  urllib3==2.2.1            # via requests
  pydantic==2.6.3           # via billing
  pydantic-core==2.16.3     # via pydantic


---
## 6. Automated Security Scanning

### pip-audit

```bash
pip install pip-audit
pip-audit -r requirements.lock
pip-audit --format json -r requirements.lock   # for CI
```

### safety

```bash
pip install safety
safety check -r requirements.lock
```

### GitHub Actions integration

```yaml
jobs:
  audit:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install pip-audit
      - run: pip-audit -r requirements.lock --format json
```

| Principle | Explanation |
|---|---|
| Scan early | Catch vulnerabilities at PR time, not after deploy |
| Scan the lock file | Pinned versions give precise, actionable results |
| Automate in CI | Manual scans are forgotten; automated ones are not |
| Treat findings as failures | A vulnerable dep should block merge, not just warn |

In [6]:
simulated_audit_result = {
    "dependencies": [
        {
            "name": "requests", "version": "2.25.0",
            "vulns": [{"id": "GHSA-j8r2-6x86-q33q", "fix_versions": ["2.31.0"],
                       "description": "Unintended leak of Proxy-Authorization header"}],
        },
        {"name": "pydantic", "version": "2.6.3", "vulns": []},
    ]
}

print("pip-audit scan results:\n")
for dep in simulated_audit_result["dependencies"]:
    status = "VULNERABLE" if dep["vulns"] else "ok"
    print(f"  {dep['name']}=={dep['version']}  [{status}]")
    for v in dep["vulns"]:
        print(f"    CVE : {v['id']}")
        print(f"    Fix : upgrade to {v['fix_versions']}")
        print(f"    Info: {v['description']}")

pip-audit scan results:

  requests==2.25.0  [VULNERABLE]
    CVE : GHSA-j8r2-6x86-q33q
    Fix : upgrade to ['2.31.0']
    Info: Unintended leak of Proxy-Authorization header
  pydantic==2.6.3  [ok]


## Module 2 — Summary

| Concept | Key Takeaway |
|---|---|
| Virtual environments | Isolate project deps from the system; always use one |
| `pyproject.toml` (PEP 621) | Single source of truth for metadata, deps, and tool config |
| Poetry | All-in-one: resolves, locks, manages venvs, and publishes |
| pip-tools | Lightweight: compiles abstract deps to a pinned lock file |
| Lock files | Guarantee reproducible installs across all machines and CI |
| pip-audit / safety | Scan lock files against CVE databases — automate in CI |

---
# Module 3 — Packaging and Advanced Distribution

## 7. Anatomy of a Modern Package

**PEP 621 is today's standard for Python packaging.** It centralizes all metadata into a single file and organizes project metadata consistently.

It replaces three legacy files:

| Replaced file | What it contained |
|---|---|
| `setup.py` | Build logic and metadata (imperative) |
| `setup.cfg` | Declarative metadata (INI format) |
| `MANIFEST.in` | Extra files to include in the sdist |

All of this now lives in `pyproject.toml` under `[project]`.

---
## 8. Setuptools, Hatch, and Flit in Action

**Use Setuptools** when you need flexibility and legacy compatibility (complex and older builds).

**Use Hatch** when you want one tool for everything: modern packaging, environment management, versioning and workflow automation.

**Use Flit** when you want to package a Python module with no configuration.

```toml
# Setuptools
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.backends.legacy:build"

# Hatch
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

# Flit
[build-system]
requires = ["flit_core>=3.2"]
build-backend = "flit_core.buildapi"
```

In [7]:
backends = {
    "setuptools": {
        "best_for": "complex builds, legacy compatibility",
        "build_backend": "setuptools.backends.legacy:build",
    },
    "hatch": {
        "best_for": "modern packaging, env management, versioning",
        "build_backend": "hatchling.build",
    },
    "flit": {
        "best_for": "simple pure-Python packages, zero config",
        "build_backend": "flit_core.buildapi",
    },
}

for name, info in backends.items():
    print(f"{name.upper()}")
    print(f"  Best for      : {info['best_for']}")
    print(f"  build-backend : {info['build_backend']}")
    print()

SETUPTOOLS
  Best for      : complex builds, legacy compatibility
  build-backend : setuptools.backends.legacy:build

HATCH
  Best for      : modern packaging, env management, versioning
  build-backend : hatchling.build

FLIT
  Best for      : simple pure-Python packages, zero config
  build-backend : flit_core.buildapi



---
## 9. Advanced Distribution Workflows

### Building distributions

```bash
pip install build
python -m build          # produces dist/billing-1.0.0.tar.gz and .whl
```

| Artefact | Extension | Description |
|---|---|---|
| sdist | `.tar.gz` | Source distribution — rebuilt on install |
| wheel | `.whl` | Pre-built binary — installs directly |

### Publishing to PyPI

```bash
pip install twine
twine upload --repository testpypi dist/*   # test first
twine upload dist/*                          # publish to PyPI
```

### Private registries

```toml
[[tool.poetry.source]]
name     = "private"
url      = "https://my.registry.example/simple/"
priority = "primary"
```

In [8]:
dist_artefacts = [
    {"file": "billing-1.0.0.tar.gz", "type": "sdist",
     "description": "Source distribution — rebuilt on target machine"},
    {"file": "billing-1.0.0-py3-none-any.whl", "type": "wheel",
     "description": "Pre-built — installs directly, no compilation needed"},
]

print("dist/ contents after `python -m build`:")
for a in dist_artefacts:
    print(f"  {a['file']:<45} [{a['type']}]")
    print(f"  └─ {a['description']}")

dist/ contents after `python -m build`:
  billing-1.0.0.tar.gz                          [sdist]
  └─ Source distribution — rebuilt on target machine
  billing-1.0.0-py3-none-any.whl                [wheel]
  └─ Pre-built — installs directly, no compilation needed


## Module 3 — Summary

| Concept | Key Takeaway |
|---|---|
| PEP 621 | Today's standard — centralizes all metadata into `pyproject.toml` |
| Setuptools | Most flexible, best for complex/legacy builds |
| Hatch | Modern all-in-one: packaging + envs + versioning |
| Flit | Simplest option — zero config for pure-Python packages |
| sdist vs wheel | Source tarball vs pre-built binary |
| twine | Publishes dist artefacts to PyPI or private registries |

---
# Module 4 — CI/CD for Scalable Project Workflows

## 10. Automating Builds and Tests

```yaml
# .github/workflows/ci.yml
name: CI
on:
  push: { branches: [main] }
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: ${{ runner.os }}-pip-${{ hashFiles('requirements.lock') }}
      - run: pip install -e '.[dev]'
      - run: pytest --tb=short
```

---
## 11. Publishing Securely with Tokens

### Trusted Publishing (OIDC) — recommended, no long-lived tokens

```yaml
# .github/workflows/publish.yml
name: Publish
on:
  push:
    tags: ["v*"]

jobs:
  publish:
    runs-on: ubuntu-latest
    permissions:
      id-token: write   # OIDC trusted publishing
    steps:
      - uses: actions/checkout@v4
      - run: pip install build
      - run: python -m build
      - uses: pypa/gh-action-pypi-publish@release/v1
        # No token needed — OIDC handles auth automatically
```

### API token fallback

```yaml
      - uses: pypa/gh-action-pypi-publish@release/v1
        with:
          password: ${{ secrets.PYPI_API_TOKEN }}
```

---
## 12. Streamlining Docs

```bash
pip install mkdocs mkdocstrings[python]
mkdocs new .
mkdocs serve          # preview at localhost:8000
mkdocs build          # generate static site in site/
```

```yaml
# mkdocs.yml
site_name: Billing Docs
plugins:
  - mkdocstrings:
      handlers:
        python:
          paths: [src]
```

Publish docs in CI:
```yaml
      - run: mkdocs gh-deploy --force   # pushes to gh-pages branch
```

In [9]:
pipeline_stages = [
    ("lint",      "ruff check src/",                "Code style and import checks"),
    ("typecheck", "mypy src/",                       "Static type verification"),
    ("test",      "pytest --tb=short",               "Unit + integration tests"),
    ("audit",     "pip-audit -r requirements.lock",  "CVE vulnerability scan"),
    ("build",     "python -m build",                 "Produce sdist + wheel"),
    ("publish",   "pypa/gh-action-pypi-publish",     "Upload to PyPI on tag push"),
    ("docs",      "mkdocs gh-deploy",                "Publish API docs to GitHub Pages"),
]

print(f"{'Stage':<12} {'Command':<40} {'Purpose'}")
print("-" * 80)
for stage, cmd, purpose in pipeline_stages:
    print(f"{stage:<12} {cmd:<40} {purpose}")

Stage        Command                                  Purpose
--------------------------------------------------------------------------------
lint         ruff check src/                          Code style and import checks
typecheck    mypy src/                                Static type verification
test         pytest --tb=short                        Unit + integration tests
audit        pip-audit -r requirements.lock           CVE vulnerability scan
build        python -m build                          Produce sdist + wheel
publish      pypa/gh-action-pypi-publish              Upload to PyPI on tag push
docs         mkdocs gh-deploy                         Publish API docs to GitHub Pages


## Module 4 — Summary

| Concept | Key Takeaway |
|---|---|
| GitHub Actions CI | Matrix builds across Python versions, with dependency caching |
| Trusted Publishing | OIDC-based PyPI auth — no long-lived tokens in secrets |
| Automated publishing | Tag push triggers build → publish pipeline |
| MkDocs + mkdocstrings | Auto-generate and host API docs from docstrings |

---
# Module 5 — Quality Gates and Type Safety at Scale

## 13. Scaling Type Checking with Mypy and Pyright

### Mypy

```bash
pip install mypy
mypy src/
```

```toml
[tool.mypy]
python_version = "3.11"
strict         = true
ignore_missing_imports = true
```

### Pyright (used by VS Code Pylance)

```bash
pip install pyright
pyright src/
```

```json
{ "include": ["src"], "typeCheckingMode": "strict" }
```

| Tool | Strengths |
|---|---|
| mypy | Mature, plugin ecosystem, strict mode |
| pyright | Faster, excellent VS Code integration, great generics support |

In [10]:
from typing import TypedDict

class Customer(TypedDict):
    id: int
    name: str
    email: str

class Invoice(TypedDict):
    customer: str
    amount: float
    status: str

def generate_invoice(customer: Customer, amount: float) -> Invoice:
    return {"customer": customer["name"], "amount": amount, "status": "pending"}

c: Customer = {"id": 1, "name": "Acme Corp", "email": "acme@example.com"}
print(generate_invoice(c, 1500.0))

# mypy/pyright would catch this at check time (not runtime):
# generate_invoice(c, "not-a-number")  # Error: str is not float

{'customer': 'Acme Corp', 'amount': 1500.0, 'status': 'pending'}


---
## 14. Automated Quality Gates in CI

```yaml
jobs:
  quality:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install ruff mypy pytest
      - run: ruff check src/           # lint
      - run: ruff format --check src/  # format check
      - run: mypy src/                 # type check
      - run: pytest --tb=short         # tests
```

All four steps must pass — a PR that breaks any gate is blocked from merging.

---
## 15. Balancing Linting, Formatting, and Type Checks

### Ruff — modern all-in-one linter + formatter

```bash
pip install ruff
ruff check src/           # lint (replaces flake8, isort, pylint rules)
ruff format src/          # format (replaces black)
ruff check --fix src/     # auto-fix safe violations
```

```toml
[tool.ruff]
line-length = 100
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "I", "UP"]   # pycodestyle, pyflakes, isort, pyupgrade
```

| Tool | Role |
|---|---|
| `ruff check` | Lint — catch style issues, unused imports, bad patterns |
| `ruff format` | Format — enforce consistent code style |
| `mypy` / `pyright` | Type check — verify type correctness statically |
| `pytest` | Test — verify behaviour at runtime |
| `pip-audit` | Security — check for known CVEs in dependencies |

In [11]:
quality_gates = [
    {"gate": "lint",     "tool": "ruff check",  "blocks_merge": True, "speed": "very fast"},
    {"gate": "format",   "tool": "ruff format", "blocks_merge": True, "speed": "very fast"},
    {"gate": "types",    "tool": "mypy",         "blocks_merge": True, "speed": "medium"},
    {"gate": "tests",    "tool": "pytest",       "blocks_merge": True, "speed": "varies"},
    {"gate": "security", "tool": "pip-audit",   "blocks_merge": True, "speed": "fast"},
]

print(f"{'Gate':<12} {'Tool':<16} {'Blocks merge':<14} {'Speed'}")
print("-" * 55)
for g in quality_gates:
    print(f"{g['gate']:<12} {g['tool']:<16} {'yes' if g['blocks_merge'] else 'no':<14} {g['speed']}")

Gate         Tool             Blocks merge   Speed
-------------------------------------------------------
lint         ruff check       yes            very fast
format       ruff format      yes            very fast
types        mypy             yes            medium
tests        pytest           yes            varies
security     pip-audit        yes            fast


## Module 5 — Summary

| Concept | Key Takeaway |
|---|---|
| mypy | Mature type checker; `strict = true` in `pyproject.toml` |
| pyright | Faster alternative; excellent VS Code / Pylance integration |
| ruff | Replaces flake8 + isort + black in one fast tool |
| Quality gates in CI | Lint → format → typecheck → test → audit; all must pass |
| `[tool.*]` in pyproject.toml | Centralise all tool config in one file |

---
# Module 6 — Extensibility and Plug-in Architectures

## 16. Extensibility in Large Systems

> *It's not about the individual tools, but the architectural mindset behind them.*

| Pillar | Description |
|---|---|
| **Modular design** | Each concern lives in its own module; no cross-cutting imports |
| **Dependency inversion** | Core code depends on an interface, not a concrete implementation |
| **Dynamic plugin loading** | Plugins are discovered at runtime, not hardcoded |
| **Entry points** | The packaging system registers plugins — no core code change needed |

This pattern is everywhere in mature Python ecosystems: `pytest` plugins, `Flask` extensions, `click` command groups, `mkdocs` plugins.

---
## 17. Designing Plug-in Hooks and Entry Points

### Simplified architecture

```
extensible_app/
├── core/
│   ├── interface.py   # Plugin contract (dependency inversion)
│   ├── loader.py      # Discovers plugins via entry points
│   └── engine.py      # Core logic that uses plugins
├── plugins/
│   └── builtin_greet.py
├── app/main.py
└── pyproject.toml     # Entry points for plugin discovery
```

### Registering plugins via pyproject.toml

```toml
[project.entry-points."extensible_app.plugins"]
greet = "extensible_app.plugins.builtin_greet:GreetPlugin"
```

### Discovering plugins at runtime

```python
from importlib.metadata import entry_points

def load_plugins(group: str) -> list:
    return [ep.load() for ep in entry_points(group=group)]
```

In [12]:
from abc import ABC, abstractmethod

# 1. Define the contract (dependency inversion)
class NotifierPlugin(ABC):
    @abstractmethod
    def notify(self, message: str) -> None: ...

    @property
    @abstractmethod
    def name(self) -> str: ...

# 2. Concrete plugins
class EmailNotifier(NotifierPlugin):
    @property
    def name(self): return "email"
    def notify(self, message): print(f"[EMAIL] {message}")

class SlackNotifier(NotifierPlugin):
    @property
    def name(self): return "slack"
    def notify(self, message): print(f"[SLACK] {message}")

# 3. Engine uses the interface, not concrete classes
def send_notifications(message: str, plugins: list[NotifierPlugin]) -> None:
    for plugin in plugins:
        plugin.notify(message)

loaded = [EmailNotifier(), SlackNotifier()]
send_notifications("Invoice #42 generated successfully", loaded)

[EMAIL] Invoice #42 generated successfully
[SLACK] Invoice #42 generated successfully


---
## 18. Building an Extensible Application Framework

### Invaluable Mindset

| Mindset | What it means |
|---|---|
| **Modules** | Keep concerns separated — billing, notifications, shared |
| **Contracts** | Define ABCs or Protocols the core depends on |
| **Extension Points** | Use entry points so plugins are added without modifying core |
| **Build for Today** | Start simple, evolve the structure |
| **Think for Tomorrow** | Design interfaces so new behaviours don't require rewrites |

### Real-world examples

| Framework | Plugin mechanism |
|---|---|
| `pytest` | `pytest11` entry point group |
| `Flask` | `init_app()` extension pattern |
| `click` | Command groups as entry points |
| `mkdocs` | `mkdocs.plugins` entry point group |

In [13]:
from importlib.metadata import entry_points

# List console_scripts entry points installed in the current environment
scripts = list(entry_points(group="console_scripts"))[:10]

print("Installed console_scripts entry points (first 10):")
for ep in scripts:
    print(f"  {ep.name:<30} → {ep.value}")

print("\nThis is exactly how a plugin system discovers extensions at runtime.")

Installed console_scripts entry points (first 10):
  hatchling                      → hatchling.cli:hatchling
  debugpy                        → debugpy.server.cli:main
  debugpy-adapter                → debugpy.adapter.__main__:main
  accelerate                     → accelerate.commands.accelerate_cli:main
  accelerate-config              → accelerate.commands.config:main
  accelerate-estimate-memory     → accelerate.commands.estimate:main
  accelerate-launch              → accelerate.commands.launch:main
  accelerate-merge-weights       → accelerate.commands.merge:main
  griffe                         → griffecli:main
  llama-index-instrumentation    → llama_index_instrumentation:main

This is exactly how a plugin system discovers extensions at runtime.


## Module 6 — Summary

| Concept | Key Takeaway |
|---|---|
| Extensibility pillars | Modular design, dependency inversion, dynamic loading, entry points |
| `[project.entry-points]` | Plugins self-register in `pyproject.toml` — no core code changes |
| `importlib.metadata` | Standard library API to discover and load registered plugins |
| ABC / Protocol contracts | Core depends on the interface, not the implementation |
| Mindset | Modules + Contracts + Extension Points; build for today, think for tomorrow |

---
# Course Summary

| Module | Topic | Core idea |
|---|---|---|
| 1 | From Scripts to Scalable Systems | `src` layout + monorepo vs multi-package |
| 2 | Dependency Management & Security | `pyproject.toml`, Poetry/pip-tools, pip-audit |
| 3 | Packaging & Advanced Distribution | PEP 621, build backends, sdist/wheel, twine |
| 4 | CI/CD for Scalable Workflows | GitHub Actions, trusted publishing, MkDocs |
| 5 | Quality Gates & Type Safety | mypy/pyright, ruff, full quality gate pipeline |
| 6 | Extensibility & Plug-in Architectures | Entry points, dependency inversion, dynamic loading |